In [26]:
import numpy as np
import dask.array as da
from dask.diagnostics import ProgressBar
import xarray as xr
import h5py

class AtmosphericDiagnostics:
    """
    Perform general atmospheric diagnostics including:
    - Anomalies
    - Eddy Momentum Flux (EMF)
    - Eddy Heat Flux (EHF)
    - Meridional EMF gradient (dEMF/dy)
    - EP Flux computation (Fj, Fk)
    - Fj [m^2/s^2]
    - Fk [Pa⋅m/s^2]
    """

    def __init__(self, u, v, t, p, z, lat=None, lon=None):
        self.u = u
        self.v = v
        self.t = t
        self.p = p
        self.z = z
        self.g = 9.81

        self.time_dim, self.z_dim, self.y_dim, self.x_dim = u.shape
        self.lat = lat if lat is not None else np.linspace(-90, 90, self.y_dim)
        self.lon = lon if lon is not None else np.linspace(0, 360, self.x_dim)

    @staticmethod
    def anomaly(data, axis=-1):
        mean = np.mean(data, axis=axis, keepdims=True)
        return data - mean

    @staticmethod
    def compute_if_dask(arr):
        return arr.compute() if isinstance(arr, da.Array) else arr

    @staticmethod
    def EMF_dask(u, v, axis=-1):
        u_prime = u - u.mean(axis=axis, keepdims=True)
        v_prime = v - v.mean(axis=axis, keepdims=True)
        return u_prime * v_prime

    def cal_theta(self):
        Rd, Cp = 287.0, 1004.0
        exponent = Rd / Cp
        theta = self.t * (100000 / self.p) ** exponent
        return theta

    def EHF_dask(self, t, p, v, axis=-1):
        theta = self.cal_theta()
        theta_mean = theta.mean(axis=axis, keepdims=True)
        theta_prime = theta - theta_mean
        v_prime = v - v.mean(axis=axis, keepdims=True)
        return v_prime * theta_prime

    def dEMF_dy(self, emf, use_dask=True, progress=True, chunks=(10, 20, 64, 128)):
        emf_dask = da.from_array(emf, chunks=chunks) if not isinstance(emf, da.Array) else emf
        R = 6.371e6
        lat_rad = np.radians(self.lat)
        dy = np.gradient(lat_rad) * R
        dy_dask = da.from_array(dy, chunks=(chunks[2],))
        dy_broadcast = dy_dask[None, None, :, None]
        d_emf_dy = da.gradient(emf_dask, axis=2) / dy_broadcast

        if use_dask:
            if progress:
                with ProgressBar():
                    return d_emf_dy.compute()
            else:
                return d_emf_dy
        else:
            emf_np = emf.compute() if isinstance(emf, da.Array) else emf
            dy_np = dy[np.newaxis, :, np.newaxis]
            return np.gradient(emf_np, axis=2) / dy_np

    def run_emf_ehf(self, chunks=(10, 20, 64, 128), progress=True):
        u_dask = da.from_array(self.u, chunks=chunks) if not isinstance(self.u, da.Array) else self.u
        v_dask = da.from_array(self.v, chunks=chunks) if not isinstance(self.v, da.Array) else self.v
        t_dask = da.from_array(self.t, chunks=chunks) if not isinstance(self.t, da.Array) else self.t
        p_dask = da.from_array(self.p, chunks=chunks) if not isinstance(self.p, da.Array) else self.p

        emf_lazy = self.EMF_dask(u_dask, v_dask)
        ehf_lazy = self.EHF_dask(t_dask, p_dask, v_dask)

        if progress:
            with ProgressBar():
                emf = emf_lazy.compute()
                ehf = ehf_lazy.compute()
        else:
            emf = emf_lazy.compute()
            ehf = ehf_lazy.compute()

        return emf, ehf

    def Cal_dtheta_dp(self):
        theta = self.cal_theta()
        p = self.p
        theta_zm = np.nanmean(theta, axis=(0, 3))
        p_zm = np.nanmean(p, axis=(0, 2, 3))
        dtheta_dp = np.zeros_like(theta_zm)
        # for k in range(1, self.z_dim - 1):
        #     dtheta_dp[k, :] = (theta_zm[k+1, :] - theta_zm[k-1, :]) / (p_zm[k+1] - p_zm[k-1])
        # dtheta_dp[0, :] = (theta_zm[1, :] - theta_zm[0, :]) / (p_zm[1] - p_zm[0])
        # dtheta_dp[-1, :] = (theta_zm[-1, :] - theta_zm[-2, :]) / (p_zm[-1] - p_zm[-2])
        for k in range(1, self.z_dim - 1):
            dtheta_dp[k, :] = (theta_zm[k-1, :] - theta_zm[k+1, :]) / (p_zm[k-1] - p_zm[k+1])
        
        # boundary (top)
        dtheta_dp[0, :] = (theta_zm[0, :] - theta_zm[1, :]) / (p_zm[0] - p_zm[1])
        # boundary (bottom)
        dtheta_dp[-1, :] = (theta_zm[-2, :] - theta_zm[-1, :]) / (p_zm[-2] - p_zm[-1])
        dtheta_dp[dtheta_dp == 0] = np.nan
        return dtheta_dp

    def f0(self):
        omega = 7.2921e-5
        return 2 * omega * np.sin(np.deg2rad(self.lat))

    def ep_flux(self, axis=-1, chunks=(10, 20, 64, 128)):
        emf, ehf = self.run_emf_ehf()
        emf_zm = np.mean(emf, axis=(0,-1))
        ehf_zm = np.mean(ehf, axis=(0,-1))
        dtheta_dp = self.Cal_dtheta_dp()
        f0_array = self.f0()[np.newaxis, :]
        F_j = -emf_zm
        # dtheta_dp_3d = dtheta_dp[np.newaxis, :, :]
        # with np.errstate(divide='ignore', invalid='ignore'):
        F_k = f0_array * ehf_zm / dtheta_dp
        return F_j, F_k

## Lag 15~25 days

In [ ]:
# import numpy as np
# import h5py
# import os
# from dask.diagnostics import ProgressBar

# PR_list = [0, 10, 20, 30, 40, 50]
# lag_range = range(15 * 4, 25 * 4)  # lag = 60 to 96 (6-hourly)

# for PR in PR_list:
#     print(f"\n==== Processing PR{PR} ====")

#     # === 一次讀入完整資料 ===
#     base_dir = f"/data92/PeterChang/back_to_master1220/Moist_Dycore/IdealizeSpetral.jl/exp/HSt42/6hourly_uv_prime_EMF/PR{PR}"
#     print("---1---")
#     with h5py.File(f"{base_dir}/u/PR{PR}_500_20000day_6hourly_u.dat", "r") as f:
#         u = f["u"][:]
#     print("---2---")
#     with h5py.File(f"{base_dir}/v/PR{PR}_500_20000day_6hourly_v.dat", "r") as f:
#         v = f["v"][:]
#     print("---3---")
#     with h5py.File(f"{base_dir}/t/PR{PR}_500_20000day_6hourly_t.dat", "r") as f:
#         t = f["t"][:]
#     print("---4---")
#     with h5py.File(f"{base_dir}/p/PR{PR}_500_20000day_6hourly_p.dat", "r") as f:
#         p = f["p"][:]
#     print("---5---")
#     with h5py.File(f"{base_dir}/z/PR{PR}_500_20000day_6hourly_z.dat", "r") as f:
#         z = f["z"][:]
#     print("---6---")
    
#     # 預先建立 adiag_all
#     adiag_all = AtmosphericDiagnostics(u, v, t, p, z)

#     # 載入該 PR 的時間 index
#     time_idx_pos_path = f"New_PC1_time_idx/PC1_positive_1std_PR{PR}.npz"
#     time_idx_neg_path = f"New_PC1_time_idx/PC1_negative_1std_PR{PR}.npz"

#     time_idx_array_positive = np.load(time_idx_pos_path)["time_idx_pos"]
#     time_idx_array_negative = np.load(time_idx_neg_path)["time_idx_neg"]

#     # 儲存 Positive & Negative 結果
#     for case_name, time_idx_array in zip(["positive", "negative"], [time_idx_array_positive, time_idx_array_negative]):
#         Fj_list, Fk_list = [], []

#         for lag in lag_range:
#             time_idx_lag = time_idx_array + lag
#             time_idx_lag = time_idx_lag[(time_idx_lag >= 0) & (time_idx_lag < u.shape[0])]

#             if len(time_idx_lag) == 0:
#                 print(f"  [{case_name.title()}] Lag {lag}: no valid indices.")
#                 continue

#             print(f"  [{case_name.title()}] Lag {lag}: {len(time_idx_lag)} samples")

#             # 切出時間索引範圍的子資料
#             adiag = AtmosphericDiagnostics(
#                 u[time_idx_lag], v[time_idx_lag], t[time_idx_lag], p[time_idx_lag], z[time_idx_lag]
#             )

#             with ProgressBar():
#                 Fj, Fk = adiag.ep_flux()

#             Fj_list.append(Fj)
#             Fk_list.append(Fk)

#         if Fj_list:
#             Fj_ens = np.mean(np.stack(Fj_list, axis=0), axis=0)
#             Fk_ens = np.mean(np.stack(Fk_list, axis=0), axis=0)

#             save_path = f"EPflux_{case_name}_lag15_25_PR{PR}.npz"
#             np.savez_compressed(save_path, Fj=Fj_ens.astype(np.float32), Fk=Fk_ens.astype(np.float32))
#             print(f"  → Saved to {save_path}")
#         else:
#             print(f"  → No data saved for {case_name} case.")



## Lag 90~100 day

In [ ]:
# import numpy as np
# import h5py
# import os
# from dask.diagnostics import ProgressBar

# PR_list = [0, 10, 20, 30, 40, 50]
# lag_range = range(90 * 4, 100 * 4)  # lag = 90 to 100 (6-hourly)

# for PR in PR_list:
#     print(f"\n==== Processing PR{PR} ====")

#     # === 一次讀入完整資料 ===
#     base_dir = f"/data92/PeterChang/back_to_master1220/Moist_Dycore/IdealizeSpetral.jl/exp/HSt42/6hourly_uv_prime_EMF/PR{PR}"
#     print("---1---")
#     with h5py.File(f"{base_dir}/u/PR{PR}_500_20000day_6hourly_u.dat", "r") as f:
#         u = f["u"][:]
#     print("---2---")
#     with h5py.File(f"{base_dir}/v/PR{PR}_500_20000day_6hourly_v.dat", "r") as f:
#         v = f["v"][:]
#     print("---3---")
#     with h5py.File(f"{base_dir}/t/PR{PR}_500_20000day_6hourly_t.dat", "r") as f:
#         t = f["t"][:]
#     print("---4---")
#     with h5py.File(f"{base_dir}/p/PR{PR}_500_20000day_6hourly_p.dat", "r") as f:
#         p = f["p"][:]
#     print("---5---")
#     with h5py.File(f"{base_dir}/z/PR{PR}_500_20000day_6hourly_z.dat", "r") as f:
#         z = f["z"][:]
#     print("---6---")

#     # 預先建立 adiag_all（非必要但保留）
#     adiag_all = AtmosphericDiagnostics(u, v, t, p, z)

#     # 載入該 PR 的時間 index
#     time_idx_pos_path = f"New_PC1_time_idx/PC1_positive_1std_PR{PR}.npz"
#     time_idx_neg_path = f"New_PC1_time_idx/PC1_negative_1std_PR{PR}.npz"

#     time_idx_array_positive = np.load(time_idx_pos_path)["time_idx_pos"]
#     time_idx_array_negative = np.load(time_idx_neg_path)["time_idx_neg"]

#     # 儲存 Positive & Negative 結果
#     for case_name, time_idx_array in zip(["positive", "negative"], [time_idx_array_positive, time_idx_array_negative]):
#         Fj_list, Fk_list = [], []

#         for lag in lag_range:
#             time_idx_lag = time_idx_array + lag
#             time_idx_lag = time_idx_lag[(time_idx_lag >= 0) & (time_idx_lag < u.shape[0])]

#             if len(time_idx_lag) == 0:
#                 print(f"  [{case_name.title()}] Lag {lag}: no valid indices.")
#                 continue

#             print(f"  [{case_name.title()}] Lag {lag}: {len(time_idx_lag)} samples")

#             # 切出時間索引範圍的子資料
#             adiag = AtmosphericDiagnostics(
#                 u[time_idx_lag], v[time_idx_lag], t[time_idx_lag], p[time_idx_lag], z[time_idx_lag]
#             )

#             with ProgressBar():
#                 Fj, Fk = adiag.ep_flux()

#             Fj_list.append(Fj)
#             Fk_list.append(Fk)

#         if Fj_list:
#             Fj_ens = np.mean(np.stack(Fj_list, axis=0), axis=0)
#             Fk_ens = np.mean(np.stack(Fk_list, axis=0), axis=0)

#             save_path = f"EPflux_{case_name}_lag90_100_PR{PR}.npz"
#             np.savez_compressed(save_path, Fj=Fj_ens.astype(np.float32), Fk=Fk_ens.astype(np.float32))
#             print(f"  → Saved to {save_path}")
#         else:
#             print(f"  → No data saved for {case_name} case.")


# All time EP flux mean

In [ ]:
import numpy as np
import h5py
from dask.diagnostics import ProgressBar

PR_list = [0, 10, 20, 30, 40, 50]
chunk_size = 1000

for PR in PR_list:
    print(f"\n==== Computing all-time EP flux for PR{PR} in chunks ====")

    base_dir = f"/data92/PeterChang/back_to_master1220/Moist_Dycore/IdealizeSpetral.jl/exp/HSt42/6hourly_uv_prime_EMF/PR{PR}"

    with h5py.File(f"{base_dir}/u/PR{PR}_500_20000day_6hourly_u.dat", "r") as f: u = f["u"][:]
    with h5py.File(f"{base_dir}/v/PR{PR}_500_20000day_6hourly_v.dat", "r") as f: v = f["v"][:]
    with h5py.File(f"{base_dir}/t/PR{PR}_500_20000day_6hourly_t.dat", "r") as f: t = f["t"][:]
    with h5py.File(f"{base_dir}/p/PR{PR}_500_20000day_6hourly_p.dat", "r") as f: p = f["p"][:]
    with h5py.File(f"{base_dir}/z/PR{PR}_500_20000day_6hourly_z.dat", "r") as f: z = f["z"][:]

    time_len = u.shape[0]
    Fj_chunks, Fk_chunks = [], []

    for t_start in range(0, time_len, chunk_size):
        t_end = min(t_start + chunk_size, time_len)
        print(f"  → Chunk {t_start}–{t_end}")

        u_chunk = u[t_start:t_end]
        v_chunk = v[t_start:t_end]
        t_chunk = t[t_start:t_end]
        p_chunk = p[t_start:t_end]
        z_chunk = z[t_start:t_end]

        adiag = AtmosphericDiagnostics(u_chunk, v_chunk, t_chunk, p_chunk, z_chunk)

        with ProgressBar():
            Fj, Fk = adiag.ep_flux()

        Fj_chunks.append(Fj)
        Fk_chunks.append(Fk)

    # === 時間平均 over all chunks ===
    Fj_mean = np.mean(np.stack(Fj_chunks, axis=0), axis=0)
    Fk_mean = np.mean(np.stack(Fk_chunks, axis=0), axis=0)

    save_path = f"EPflux_alltime_PR{PR}.npz"
    np.savez_compressed(save_path, Fj=Fj_mean.astype(np.float32), Fk=Fk_mean.astype(np.float32))
    print(f"  → Saved mean EP flux to {save_path}")
